Every pipeline notebook starts with `%run ./00_common`. 
This notebook has the settings and helper functions.

Schema type and what it holds:
1. Bronze: Exact copies of the four CSVs, every value as text, plus lineage columns
2. Silver: Cleaned, typed, conformed tables, one per source, plus crosswalks and the event match (Applies rules from the data quality log)
3. Gold: Star schema (dimensions and facts)
4. Audit: Run log, rejects, file manifest, validation results (append only hsitory)

In [0]:
import os, re, uuid, json, html, hashlib
from datetime import datetime
import pandas as pd
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
if not IS_DATABRICKS:
    spark = (SparkSession.builder.master("local[2]").appName("disaster-dw")
             .config("spark.ui.enabled", "false")
             .config("spark.sql.execution.arrow.pyspark.enabled", "true")
             .config("spark.sql.session.timeZone", "Australia/Sydney")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")

CATALOG = "Workspace" if IS_DATABRICKS else "spark_catalog"
RAW_PATH = "/Workspace/Users/raahatpathak123@gmail.com/disaster-dw/data/raw/" if IS_DATABRICKS else "./data/raw"
TABLE_FORMAT = "delta" if IS_DATABRICKS else "parquet"
FILES = {"ica": "ica.csv", "disaster_events": "disaster_events.csv",
         "abs_population": "abs_population.csv", "abs_income": "abs_income.csv"}
HEADER_ROW = {"ica": 3, "disaster_events": 0, "abs_population": 6, "abs_income": 6}   # rows above the real header (see profiling)

SCOPE_START = "2011-01-01"          # P1 scope decision
SCOPE_END_YEAR = 2026               # last year in dim_date / fact_state_year
DATE_SENTINEL = "1753-01-01"        # SQL Server DATETIME minimum, used as a fake date in disaster events

def get_param(name, default=""):
    """Notebook parameter: a widget on Databricks, an environment variable locally."""
    if IS_DATABRICKS:
        try:
            dbutils.widgets.text(name, default)
            return dbutils.widgets.get(name) or default
        except Exception:
            return default
    return os.environ.get(name.upper(), default)

RUN_ID = get_param("run_id") or datetime.now().strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:6]

try:
    display  # Databricks provides display()
except NameError:
    def display(obj, n=25):
        print(obj.limit(n).toPandas().to_string() if isinstance(obj, DataFrame) else obj)

for schema in ["bronze", "silver", "gold", "audit"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")

print(f"run_id={RUN_ID} | catalog={CATALOG} | raw={RAW_PATH} | format={TABLE_FORMAT}")

### Table helpers

In [0]:
def tname(layer, name):
    return f"{CATALOG}.{layer}.{name}"

def read(layer, name):
    return spark.table(tname(layer, name))

def write(df, layer, name, mode="overwrite"):
    """Write a table and return its row count."""
    (df.write.format(TABLE_FORMAT).mode(mode).option("overwriteSchema", "true").saveAsTable(tname(layer, name)))
    return spark.table(tname(layer, name)).count()

def with_unknown(df, **values):
    """Append the Unknown member (key -1) to a dimension. Unspecified columns are NULL."""
    schema = T.StructType([T.StructField(f.name, f.dataType, True) for f in df.schema])
    row = tuple(values.get(f.name) for f in schema)
    return df.unionByName(spark.createDataFrame([row], schema))

def add_constraints(statements):
    """Informational PK/FK constraints (Unity Catalog only). They document the model; validation enforces it."""
    if not IS_DATABRICKS:
        print("Local run: constraints skipped (Unity Catalog feature).")
        return
    for sql in statements:
        try:
            spark.sql(sql)
        except Exception as e:
            print("Constraint skipped:", sql[:80], "->", str(e).splitlines()[0][:120])

def table_exists(layer, name):
    return spark.catalog.tableExists(tname(layer, name))

### Audit helpers
- `log_step` writes one row per load step: rows in, rows out, rows rejected.
- `reject` writes rows that fail a rule, with the rule id and the full original record as JSON. Nothing is deleted without a trace.

In [0]:
RUN_LOG_SCHEMA = "run_id string, layer string, step string, target string, rows_in bigint, rows_out bigint, rows_rejected bigint, status string, note string, logged_at timestamp"
REJECT_SCHEMA = "run_id string, layer string, source string, source_row int, rule_id string, reason string, raw_record string, rejected_at timestamp"

def log_step(layer, step, target, rows_in, rows_out, rows_rejected=0, note=""):
    status = "OK" if rows_in == rows_out + rows_rejected else "CHECK"
    row = [(RUN_ID, layer, step, target, int(rows_in), int(rows_out), int(rows_rejected), status, note, datetime.now())]
    spark.createDataFrame(row, RUN_LOG_SCHEMA).write.format(TABLE_FORMAT).mode("append").saveAsTable(tname("audit", "run_log"))
    print(f"[{status}] {layer}.{step}: in={rows_in:,} out={rows_out:,} rejected={rows_rejected:,} {note}")

def reject(df, layer, source, rule_id, reason_col_or_text, row_col="_row_num"):
    """Append rejected rows to audit.rejects. Returns the number rejected."""
    reason = reason_col_or_text if isinstance(reason_col_or_text, type(F.lit(1))) else F.lit(reason_col_or_text)
    payload = [c for c in df.columns if not c.startswith("_")]
    out = df.select(F.lit(RUN_ID).alias("run_id"), F.lit(layer).alias("layer"), F.lit(source).alias("source"),
                    F.col(row_col).cast("int").alias("source_row"), F.lit(rule_id).alias("rule_id"), reason.alias("reason"),
                    F.to_json(F.struct(*payload)).alias("raw_record"), F.current_timestamp().alias("rejected_at"))
    n = out.count()
    if n:
        out.write.format(TABLE_FORMAT).mode("append").saveAsTable(tname("audit", "rejects"))
    return n

### Cleaning rules as reusable expressions
Each function is one rule from the data-quality log, so the same rule is applied the same way everywhere.

In [0]:
def clean_name(c):
    c = re.sub(r"[^0-9a-z]+", "_", str(c).strip().lower()).strip("_")
    return f"c_{c}" if c[:1].isdigit() else (c or "unnamed")

def num(col, dash_is_zero):
    """DQ3: text -> decimal. Strips $, commas and spaces. '-' / '$-' is accounting nil: 0 (ICA) or NULL (ABS 'not available')."""
    t = F.regexp_replace(F.trim(F.col(col)), r"[$,\s]", "")
    dash = F.lit(0).cast("decimal(18,2)") if dash_is_zero else F.lit(None).cast("decimal(18,2)")
    return F.when(F.col(col).isNull() | (t == ""), F.lit(None).cast("decimal(18,2)")) \
            .when(t == "-", dash) \
            .otherwise(F.expr(f"try_cast(regexp_replace(trim(`{col}`), '[$,\\\\s]', '') AS DECIMAL(18,2))"))

def parse_date(col, fmt):
    return F.to_date(F.expr(f"try_to_timestamp(trim(`{col}`), '{fmt}')"))

def fix_century(d):
    """DQ1: two-digit years land 100 years late (1967 -> 2067). Pull any future date back a century."""
    return F.when(F.year(d) > F.year(F.current_date()), F.add_months(d, -1200)).otherwise(d)

def blank_to_null(col):
    return F.when(F.trim(F.col(col)) == "", None).otherwise(F.trim(F.col(col)))

unescape_html = F.udf(lambda s: html.unescape(s) if s is not None else None, T.StringType())   # DQ11